# What our TinyStories budget actually purchased

Chapter 6 · Day 9

The model consumed 48.84M valid targets while processing 229.38M padded positions. Compare measured clocks and loss from the real archived run; avoid loading weights or starting a server.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').exists())
sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(909)
plt.rcParams.update({'figure.figsize': (8, 3.6), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})

In [ ]:
from dongxi_llms.course_foundations import budget_summary
e = json.loads((ROOT / 'experiments/reports/2026-09-14-tinystories-learning-result.json').read_text())
s = budget_summary(e)
print(json.dumps(s, indent=2))
assert e['completion']['completed_updates'] == 14000
assert abs(s['valid_fraction'] - .2129253932) < 1e-8

Prediction: why do update-timer throughput and total-run throughput differ when they count the same valid labels? Reference: the elapsed-time scopes differ. Preparation is outside both. The ratio of sums weights longer updates correctly; a mean of heterogeneous per-batch ratios answers another question.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].bar(['Processed', 'Valid', 'Prepared'], [e['metrics_summary']['processed_positions']/1e6, e['metrics_summary']['valid_targets']/1e6, e['data_manifest']['train']['valid_targets']/1e6], color=['#adb4bd','#3265ad','#a56928'])
axes[0].set(ylabel='Positions / target presentations (millions)', title='Different counts, different questions')
axes[1].plot([r['update'] for r in e['validation_curve']], [r['loss'] for r in e['validation_curve']], color='#3265ad')
axes[1].set(xlabel='Completed updates', ylabel='Fixed-development NLL (nats)')
fig.tight_layout()
plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![01 read training clocks: reference plot 01](../figures/chapter-06/day-09-01_read_training_clocks-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

Reference explanation: prepared targets describe availability; consumed labels describe exposure; processed positions include padding. They are not three estimates of unique linguistic knowledge. The two panels use actual stored counts and loss, not extrapolated training progress.

Intervention: calculate token rate if someone incorrectly divides processed positions by total runtime. Reference: it becomes a larger positions/s figure, but reporting it as valid targets/s is a unit error. Changing the denominator can create the appearance of faster learning without changing a single update.

What remains: a systems change such as length-aware batching needs an actual controlled run; the archive alone cannot measure its speedup. See the next notebook for attention-boundary invariants that such a change must retain.

## A restart restores weights, not unused work

Prediction: an actual tiny Session completes 9 valid targets, saves, then fails during an admitted 8-target update. A fresh Session restores that boundary and retries the same group. Should successful exposure and reserved work both be 17? Should changing the output directory erase the failed attempt?

The companion below uses the original native causal decoder, token-weighted loss, AdamW, schedule and authored ID windows. It is a small CPU source control, separate from the archived 66.64M TinyStories run and not a story-quality experiment. Whole model operations are reserved; data planning, byte I/O, serialization and physical resource limits are not covered.

In [ ]:
from dongxi_llms.stories_work_lab import work_recovery_example
attempt_target_cap = 25  # Change only this argument to 24, then rerun both cells.
work = work_recovery_example(attempt_target_cap)
if attempt_target_cap == 25:
    assert (work['successful_targets'], work['reserved_targets']) == (17, 25)
    assert work['exact_numerical_recovery'] and not work['retry_refused']
else:
    assert (work['successful_targets'], work['reserved_targets']) == (9, 17)
    assert work['retry_refused']
assert work['later_failed_work_retained'] and work['next_refusal_forward_calls'] == 0
assert work['refusal_numerical_state_unchanged'] and work['refusal_work_prefix_unchanged']
print('Successful labels:', work['successful_targets'], 'Reserved target places:', work['reserved_targets'])
print('Successful updates:', work['actual_successful_updates'], 'Original schedule horizon:', work['recipe_horizon'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
events = work['timeline']
x = list(range(len(events)))
axes[0].plot(x, [r['successful_targets'] for r in events], 'o-', color='#3265ad', label='Successful training labels')
axes[0].plot(x, [r['reserved_targets'] for r in events], 's--', color='#a56928', label='Reserved target places')
last_label = 'Refuse 8' if work['retry_refused'] else 'Retry 8'
axes[0].set(xticks=x, xticklabels=['Complete 9', 'Fail 8', 'Restore 9', last_label], ylim=(0, 28), ylabel='Target labels / reserved places', title='Numerical state rewinds; spent work does not')
axes[0].legend(frameon=False, fontsize=9)
keys = ['policy_forward_calls', 'backward_calls', 'optimizer_calls']
labels = ['Forward', 'Backward', 'Optimizer']
for offset, name, color, label in [(-.3, 'reserved', '#adb4bd', 'Reserved'), (-.1, 'completed', '#3265ad', 'Completed'), (.1, 'known_partial', '#a56928', 'Known partial'), (.3, 'uncertain_upper', '#af4448', 'Uncertain upper')]:
    axes[1].bar([i+offset for i in range(3)], [work[name][k] for k in keys], width=.18, color=color, label=label)
axes[1].set(xticks=range(3), xticklabels=labels, ylabel='Declared logical calls', title='A failed call is not a completed update')
axes[1].legend(frameon=False, fontsize=9, ncols=2)
fig.tight_layout()
plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Actual tiny story Session: successful targets and retained work diverge after a failed attempt](../figures/chapter-06/day-09-01_read_training_clocks-02.png)

This new preview comes only from the native CPU companion; the earlier plot still shows the historical TinyStories observations.

Reference: at capacity25, the first successful group and retry learn from 9+8=17 labels, but admitted groups reserve 9+8+8=25 places. The failed first forward/backward is real work; the fresh restore preserves its journal suffix while restoring the old weights, moments and data stream. The original two-update numerical state is exactly reproduced. The next complete group cannot fit the exhausted attempted-target cap, so it makes no forward call and changes no numerical state. The original five-update schedule was not shortened; this microscope deliberately stops after two successes.

Controlled change: set the notebook's attempt_target_cap argument to24 and rerun both cells. The same complete eight-target retry must be refused with only7 places available; do not remove a label to make it fit. Predict which clock stays at9 and which remains at17. Return the argument to25 to permit the retry in a fresh independently bound journal. This changes an admission experiment, not the loss or labels, and grants no GPU run. A refused retry is not a successful two-update final replay.

An entered optimizer failure can leave half-mutated weights. The accounted Session is then unusable for updates, observations or publication; only a fresh restored Session may continue. Successful recovery is neither improved story quality nor physical containment. See Chapter 6, worked answer20 and the persistent-work specification.